In [1]:

"""SESSION 34: YOLO26n-seg source-grouped CV on Image_Annotation_rebuilt."""
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import multiprocessing
import random
import re
import statistics
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


SESSION_NAME = "SESSION_34"
MODEL_NAME = "yolo26n-seg.pt"
FOLD_COUNT = 5
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMAGE_ANNOTATION_CLASSES = ["Nail_psoriasis", "SJS-TEN", "acne", "vitiligo"]
AUGMENT_PREFIX = re.compile(
    r"^(?:"
    r"Crop-(?:100450_100400|100450_0300|100350_100300|0350_100400|0350_0300)|"
    r"padd-(?:100100_100100|1000_00|0100_00|00_1000|00_0100)|"
    r"Resize-450_400|"
    r"flip-(?:-1|0|1)|"
    r"(?:hue|saturation)-(?:200|150|100|50)|"
    r"superpixels-(?:200|100|75|50|25)"
    r")"
)
SPLIT_ALIASES = {
    "train": ("train",),
    "val": ("valid", "val", "validation"),
    "test": ("test",),
}


def _ensure_dependencies() -> None:
    print("[install] Ensuring the latest Ultralytics package for YOLO26 support.", flush=True)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "ultralytics"]
    )
    import ultralytics
    version = getattr(ultralytics, "__version__", "unknown")
    print(f"[environment] ultralytics={version}", flush=True)


def _has_split(root: Path, split: str) -> bool:
    return any(
        (root / name / "images").is_dir() and (root / name / "labels").is_dir()
        for name in SPLIT_ALIASES[split]
    )


def _valid_dataset_root(root: Path) -> bool:
    return (
        (root / "data.yaml").is_file()
        and all(_has_split(root, split) for split in SPLIT_ALIASES)
    )


def _has_skin_disnet_manifest(root: Path) -> bool:
    manifest = root / "manifest.csv"
    if not manifest.is_file():
        return False
    with manifest.open(newline="", encoding="utf-8") as stream:
        columns = set(next(csv.reader(stream), []))
    return {"name", "split", "source", "augmented"} <= columns


def _is_image_annotation_dataset(root: Path) -> bool:
    yaml_path = root / "data.yaml"
    if not yaml_path.is_file():
        return False
    text = yaml_path.read_text(encoding="utf-8")
    return all(re.search(rf"^\s*\d+\s*:\s*{re.escape(name)}\s*$", text, re.MULTILINE)
               for name in IMAGE_ANNOTATION_CLASSES)


def _resolve_dataset_root(hint: str) -> Path:
    if hint != "auto":
        explicit = Path(hint).expanduser().resolve()
        if _valid_dataset_root(explicit):
            return explicit
        if explicit.is_dir():
            nested = sorted(
                {
                    path.parent.resolve()
                    for path in explicit.rglob("data.yaml")
                    if _valid_dataset_root(path.parent)
                }
            )
            preferred = [path for path in nested if _is_image_annotation_dataset(path)]
            preferred += [
                path for path in nested
                if path not in preferred and (
                    _has_skin_disnet_manifest(path)
                    or path.name.lower().replace("-", "_") == "skin_disease_yolo"
                )
            ]
            if len(preferred) == 1:
                return preferred[0]
            if len(nested) == 1:
                return nested[0]
        raise FileNotFoundError(
            f"Dataset not found or incomplete at {explicit}; expected data.yaml "
            "and train/valid/test image and label directories."
        )

    search_roots = [Path("/kaggle/input"), Path.cwd()]
    candidates: set[Path] = set()
    for search_root in search_roots:
        if not search_root.is_dir():
            continue
        for yaml_path in search_root.rglob("data.yaml"):
            root = yaml_path.parent.resolve()
            if _valid_dataset_root(root):
                candidates.add(root)

    if not candidates:
        raise FileNotFoundError(
            "Could not locate a supported YOLO segmentation dataset. Attach "
            "Image_Annotation_rebuilt in Kaggle or pass --data-root explicitly."
        )

    rebuilt = sorted(root for root in candidates if _is_image_annotation_dataset(root))
    manifested = [root for root in candidates if _has_skin_disnet_manifest(root)]
    named = [
        root for root in candidates
        if root.name.lower().replace("-", "_") == "skin_disease_yolo"
    ]
    selected = rebuilt or manifested or named or sorted(candidates)
    if len(selected) != 1:
        choices = "\n".join(f"  - {path}" for path in selected)
        raise RuntimeError(
            "Found multiple candidate YOLO datasets. Attach Image_Annotation_rebuilt "
            "or pass --data-root explicitly:\n"
            f"{choices}"
        )
    return selected[0]


def _load_dataset_yaml(root: Path) -> tuple[list[str], dict[str, str]]:
    try:
        import yaml
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--upgrade", "PyYAML"]
        )
        import yaml

    document = yaml.safe_load((root / "data.yaml").read_text(encoding="utf-8")) or {}
    raw_names = document.get("names")
    if isinstance(raw_names, dict):
        names = [str(raw_names[key]) for key in sorted(raw_names, key=lambda k: int(k))]
    elif isinstance(raw_names, list):
        names = [str(name) for name in raw_names]
    else:
        raise ValueError("data.yaml must define class names as a list or indexed mapping.")
    declared_count = document.get("nc")
    if declared_count is not None and int(declared_count) != len(names):
        raise ValueError(
            f"data.yaml declares nc={declared_count} but lists {len(names)} classes."
        )
    if not names:
        raise ValueError("data.yaml contains no class names.")

    split_dirs: dict[str, str] = {}
    for output_name, aliases in SPLIT_ALIASES.items():
        split_name = next(
            (name for name in aliases if (root / name / "images").is_dir()),
            None,
        )
        if split_name is None:
            raise FileNotFoundError(f"Missing image directory for the {output_name} split.")
        split_dirs[output_name] = split_name
    return names, split_dirs


def _validate_labels(
    root: Path, names: list[str], split_dirs: dict[str, str]
) -> dict[str, int]:
    counts: dict[str, int] = {}
    for split, directory_name in split_dirs.items():
        images_dir = root / directory_name / "images"
        labels_dir = root / directory_name / "labels"
        images = sorted(
            path for path in images_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        )
        if not images:
            raise ValueError(f"No images found in {images_dir}")
        for image in images:
            label = labels_dir / f"{image.stem}.txt"
            if not label.is_file():
                raise FileNotFoundError(f"Missing segmentation label for {image}: {label}")
            for line_number, line in enumerate(
                label.read_text(encoding="utf-8").splitlines(), start=1
            ):
                fields = line.split()
                if not fields:
                    continue
                if len(fields) < 7 or len(fields) % 2 == 0:
                    raise ValueError(
                        f"Expected class and at least three (x,y) points in "
                        f"{label}:{line_number}"
                    )
                try:
                    values = [float(value) for value in fields]
                except ValueError as error:
                    raise ValueError(f"Non-numeric segmentation label at {label}:{line_number}") from error
                class_id = int(values[0])
                if values[0] != class_id or not 0 <= class_id < len(names):
                    raise ValueError(
                        f"Class id {values[0]} is outside [0, {len(names) - 1}] "
                        f"at {label}:{line_number}"
                    )
                coordinates = values[1:]
                if not all(0.0 <= coordinate <= 1.0 for coordinate in coordinates):
                    raise ValueError(
                        f"Segmentation coordinates must be normalized to [0, 1] "
                        f"at {label}:{line_number}"
                    )
        counts[split] = len(images)
    return counts


def _collect_cv_items(
    root: Path,
    split_dirs: dict[str, str],
    group_filename_augmentations: bool = False,
) -> list[dict[str, Any]]:
    source_manifest: dict[tuple[str, str], str] = {}
    manifest_path = root / "manifest.csv"
    if _has_skin_disnet_manifest(root):
        with manifest_path.open(newline="", encoding="utf-8") as stream:
            reader = csv.DictReader(stream)
            for row in reader:
                split = row.get("split", "").strip().lower()
                split = "val" if split in {"valid", "validation"} else split
                name = Path(row.get("name", "")).stem
                source = row.get("source", "").strip()
                key = (split, name)
                if split not in SPLIT_ALIASES or not name or not source or key in source_manifest:
                    raise ValueError(
                        f"Invalid or duplicate row in source manifest {manifest_path}: {row}"
                    )
                source_manifest[key] = source

    items = []
    discovered: set[tuple[str, str]] = set()
    source_splits: dict[str, set[str]] = {}
    items_by_group: dict[str, list[int]] = {}
    for split in ("train", "val", "test"):
        directory_name = split_dirs[split]
        images_dir = root / directory_name / "images"
        labels_dir = root / directory_name / "labels"
        for image in sorted(
            path for path in images_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        ):
            source_key = (split, image.stem)
            source = source_manifest.get(source_key)
            if source_manifest and source is None:
                raise ValueError(
                    f"Image {image} is missing from source manifest {manifest_path}."
                )
            if source_manifest:
                discovered.add(source_key)
            if group_filename_augmentations:
                raw_stem = image.stem.split(".rf.", 1)[0]
                source = AUGMENT_PREFIX.sub("", raw_stem, count=1)
                group = source
            else:
                if source:
                    source_splits.setdefault(source, set()).add(split)
                group = source or image.stem.split(".rf.", 1)[0]
                if split == "test":
                    continue
            label = labels_dir / f"{image.stem}.txt"
            classes = {
                int(line.split()[0])
                for line in label.read_text(encoding="utf-8").splitlines()
                if line.strip()
            }
            item = {
                "image": str(image.resolve()),
                "group": group,
                "classes": sorted(classes),
                "manifest_source": bool(source_manifest),
            }
            if group_filename_augmentations:
                item.update({
                    "source_group_cv": True,
                    "validation_image": False,
                    "augmented_filename": bool(AUGMENT_PREFIX.match(raw_stem)),
                    "original_split": split,
                })
                source_splits.setdefault(group, set()).add(split)
            items.append(item)
            items_by_group.setdefault(group, []).append(len(items) - 1)
    if source_manifest:
        missing = set(source_manifest) - discovered
        if missing:
            raise ValueError(
                f"{len(missing)} manifest entries have no matching image; "
                f"first missing entry: {sorted(missing)[0]}"
            )
        leaking_sources = [
            source for source, splits in source_splits.items()
            if "test" in splits and len(splits) > 1
        ]
        if leaking_sources and not group_filename_augmentations:
            raise ValueError(
                "Source manifest overlaps the locked test split; first source: "
                + sorted(leaking_sources)[0]
            )
    if group_filename_augmentations:
        class_conflicts = {
            group for group, indices in items_by_group.items()
            if len({tuple(items[index]["classes"]) for index in indices}) > 1
        }
        items = [item for item in items if item["group"] not in class_conflicts]
        items_by_group = {}
        for index, item in enumerate(items):
            items_by_group.setdefault(item["group"], []).append(index)
        for indices in items_by_group.values():
            candidates = [index for index in indices if not items[index]["augmented_filename"]]
            selected = min(candidates or indices, key=lambda index: items[index]["image"])
            items[selected]["validation_image"] = True
        overlap_count = sum(len(splits) > 1 for splits in source_splits.values())
        for item in items:
            item["source_split_overlap_groups"] = overlap_count
            item["excluded_source_class_conflict_groups"] = len(class_conflicts)
    if len(items) < FOLD_COUNT:
        raise ValueError(f"Five-fold CV requires at least {FOLD_COUNT} images.")
    return items


def _assign_folds(
    items: list[dict[str, Any]], n_classes: int, seed: int
) -> tuple[list[list[int]], dict[str, Any]]:
    groups: dict[str, list[int]] = {}
    class_totals = [0] * n_classes
    source_group_cv = any(item.get("source_group_cv") for item in items)
    for index, item in enumerate(items):
        groups.setdefault(item["group"], []).append(index)
        for class_id in item["classes"]:
            class_totals[class_id] += 1
    if len(groups) < FOLD_COUNT:
        raise ValueError(
            f"Five-fold group-safe CV requires at least {FOLD_COUNT} source groups; "
            f"found {len(groups)}."
        )

    source_class_totals = [0] * n_classes
    if source_group_cv:
        for indices in groups.values():
            for class_id in {c for index in indices for c in items[index]["classes"]}:
                source_class_totals[class_id] += 1
    grouped_items = []
    for group_name, indices in groups.items():
        class_counts = [0] * n_classes
        if not source_group_cv:
            for index in indices:
                for class_id in items[index]["classes"]:
                    class_counts[class_id] += 1
        if source_group_cv:
            for class_id in {c for index in indices for c in items[index]["classes"]}:
                class_counts[class_id] = 1
        rarity_totals = source_class_totals if source_group_cv else class_totals
        rarity = sum(
            (1 if source_group_cv and count else count) / max(total, 1)
            for count, total in zip(class_counts, rarity_totals)
        )
        grouped_items.append({
            "name": group_name,
            "indices": indices,
            "class_counts": class_counts,
            "rarity": rarity,
        })

    rng = random.Random(seed)
    for group in grouped_items:
        group["tie_break"] = rng.random()
    grouped_items.sort(
        key=lambda group: (
            -group["rarity"],
            0 if source_group_cv else -len(group["indices"]),
            group["tie_break"],
            group["name"],
        )
    )

    target_size = (len(groups) if source_group_cv else len(items)) / FOLD_COUNT
    balance_totals = source_class_totals if source_group_cv else class_totals
    target_class_counts = [total / FOLD_COUNT for total in balance_totals]
    fold_indices: list[list[int]] = [[] for _ in range(FOLD_COUNT)]
    fold_group_names: list[set[str]] = [set() for _ in range(FOLD_COUNT)]
    fold_class_counts = [[0] * n_classes for _ in range(FOLD_COUNT)]

    for group in grouped_items:
        costs = []
        for fold_id in range(FOLD_COUNT):
            size_after = len(fold_group_names[fold_id]) + 1 if source_group_cv else (
                len(fold_indices[fold_id]) + len(group["indices"])
            )
            size_cost = (size_after / max(target_size, 1.0)) ** 2
            class_cost = sum(
                ((fold_class_counts[fold_id][class_id] + count)
                 / max(target_class_counts[class_id], 1.0)) ** 2
                for class_id, count in enumerate(group["class_counts"])
            )
            costs.append((size_cost + class_cost, len(fold_group_names[fold_id]), fold_id))
        selected = min(costs)[2]
        fold_indices[selected].extend(group["indices"])
        fold_group_names[selected].add(group["name"])
        for class_id, count in enumerate(group["class_counts"]):
            fold_class_counts[selected][class_id] += count

    if any(not indices for indices in fold_indices):
        raise RuntimeError("The fold assignment produced an empty validation fold.")

    report = {
        "fold_count": FOLD_COUNT,
        "seed": seed,
        "image_count": len(items),
        "source_group_count": len(groups),
        "fold_image_counts": [len(indices) for indices in fold_indices],
        "fold_group_variant_image_counts": [len(indices) for indices in fold_indices],
        "fold_validation_image_counts": [
            sum(items[index].get("validation_image", True) for index in indices)
            for indices in fold_indices
        ],
        "fold_group_counts": [len(names) for names in fold_group_names],
        "fold_class_image_counts": None if source_group_cv else fold_class_counts,
        "class_image_totals": class_totals,
        "fold_class_source_group_counts": fold_class_counts if source_group_cv else None,
        "source_group_cv": source_group_cv,
        "source_class_group_totals": source_class_totals if source_group_cv else None,
        "grouping_rule": (
            "filename-derived augmentation source; all train/valid/test variants grouped; "
            "one original-preferred image per source used for validation"
            if source_group_cv
            else "manifest source lineage; source variants stay in one fold"
            if any(item.get("manifest_source") for item in items)
            else "image stem before '.rf.'; source variants stay in one fold"
        ),
    }
    return fold_indices, report


def _write_cv_fold_yamls(
    work_dir: Path,
    items: list[dict[str, Any]],
    fold_indices: list[list[int]],
    names: list[str],
    seed: int,
) -> list[Path]:
    import yaml

    all_indices = set(range(len(items)))
    yaml_paths = []
    for fold_id, valid_indices in enumerate(fold_indices):
        valid_set = set(valid_indices)
        train_indices = sorted(all_indices - valid_set)
        valid_groups = {items[index]["group"] for index in valid_set}
        valid_indices = sorted(
            index for index in valid_set
            if items[index].get("validation_image", True)
        )
        if not valid_indices:
            raise RuntimeError(f"Fold {fold_id} has no validation images.")
        if any(items[index]["group"] not in valid_groups for index in valid_indices):
            raise RuntimeError(f"Fold {fold_id} validation images do not match held-out groups.")
        for train_index in train_indices:
            if items[train_index]["group"] in valid_groups:
                raise RuntimeError(f"Source group leaked across fold {fold_id}.")

        fold_dir = work_dir / "folds" / f"seed_{seed}" / f"fold_{fold_id}"
        fold_dir.mkdir(parents=True, exist_ok=True)
        train_list = fold_dir / "train.txt"
        valid_list = fold_dir / "valid.txt"
        train_list.write_text(
            "\n".join(items[index]["image"] for index in train_indices) + "\n",
            encoding="utf-8",
        )
        valid_list.write_text(
            "\n".join(items[index]["image"] for index in valid_indices) + "\n",
            encoding="utf-8",
        )
        data_yaml = fold_dir / "data.yaml"
        data_yaml.write_text(
            yaml.safe_dump(
                {
                    "train": str(train_list.resolve()),
                    "val": str(valid_list.resolve()),
                    "nc": len(names),
                    "names": names,
                },
                sort_keys=False,
                allow_unicode=True,
            ),
            encoding="utf-8",
        )
        yaml_paths.append(data_yaml)
    return yaml_paths


def _select_device(requested: str) -> str:
    if requested != "auto":
        return requested
    try:
        import torch

        if torch.cuda.is_available():
            device = ",".join(str(index) for index in range(torch.cuda.device_count()))
            print(
                f"[device] using GPU(s) {device} of {torch.cuda.device_count()} available.",
                flush=True,
            )
            return device
    except ImportError:
        pass
    print("[device] CUDA unavailable; using CPU.", flush=True)
    return "cpu"


def _metric_value(metrics: Any, key: str, default: float | None = None) -> float | None:
    value = getattr(metrics, key, default)
    if value is None:
        return None
    try:
        value = float(value)
    except (TypeError, ValueError):
        return default
    return value if value == value else None


def _segmentation_results(metrics: Any, names: list[str]) -> dict[str, Any]:
    segmentation = getattr(metrics, "seg", None)
    if segmentation is None:
        raise RuntimeError(
            "Ultralytics did not return segmentation metrics. Confirm that the "
            "checkpoint ends in '-seg.pt' and that labels contain polygons."
        )

    per_class: dict[str, float | None] = {name: None for name in names}
    ap = getattr(segmentation, "ap", None)
    class_indices = getattr(segmentation, "ap_class_index", None)
    if ap is not None and class_indices is not None:
        ap_values = ap.tolist() if hasattr(ap, "tolist") else list(ap)
        indices = (
            class_indices.tolist()
            if hasattr(class_indices, "tolist")
            else list(class_indices)
        )
        for index, class_ap in zip(indices, ap_values):
            class_index = int(index)
            if not 0 <= class_index < len(names) or class_ap is None:
                continue
            if isinstance(class_ap, (list, tuple)) and not class_ap:
                continue
            ap50 = class_ap[0] if isinstance(class_ap, (list, tuple)) else class_ap
            if ap50 is not None:
                score = float(ap50)
                if score == score:
                    per_class[names[class_index]] = score

    return {
        "segm_mAP50": _metric_value(segmentation, "map50"),
        "segm_mAP50_95": _metric_value(segmentation, "map"),
        "per_class_segm_mAP50": per_class,
        "bbox_mAP50": _metric_value(getattr(metrics, "box", None), "map50"),
        "bbox_mAP50_95": _metric_value(getattr(metrics, "box", None), "map"),
    }


def _write_per_class_csv(path: Path, scores: dict[str, float | None]) -> None:
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=["class", "segm_mAP50"])
        writer.writeheader()
        for name, score in scores.items():
            writer.writerow({"class": name, "segm_mAP50": score})


def _completed_epoch_count(results_csv: Path) -> int:
    if not results_csv.is_file():
        return 0
    with results_csv.open(newline="", encoding="utf-8") as stream:
        return sum(1 for _ in csv.DictReader(stream))


def _aggregate_fold_metrics(
    fold_results: list[dict[str, Any]], names: list[str]
) -> dict[str, Any]:
    aggregate: dict[str, Any] = {}
    for metric_name in ("segm_mAP50", "segm_mAP50_95", "bbox_mAP50", "bbox_mAP50_95"):
        values = [
            float(result["metrics"][metric_name])
            for result in fold_results
            if result["metrics"].get(metric_name) is not None
        ]
        aggregate[metric_name] = {
            "mean": statistics.fmean(values) if values else None,
            "std": statistics.pstdev(values) if len(values) > 1 else 0.0 if values else None,
            "folds_with_metric": len(values),
        }

    per_class: dict[str, dict[str, float | int | None]] = {}
    for name in names:
        values = [
            float(result["metrics"]["per_class_segm_mAP50"][name])
            for result in fold_results
            if result["metrics"]["per_class_segm_mAP50"].get(name) is not None
        ]
        per_class[name] = {
            "mean": statistics.fmean(values) if values else None,
            "std": statistics.pstdev(values) if len(values) > 1 else 0.0 if values else None,
            "folds_with_metric": len(values),
        }
    aggregate["per_class_segm_mAP50"] = per_class
    return aggregate


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(
        description="SESSION 34: Kaggle-ready, source-grouped YOLO26n segmentation CV"
    )
    parser.add_argument("--data-root", default="auto")
    parser.add_argument("--work-dir", default=None)
    parser.add_argument("--model", default=MODEL_NAME)
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument(
        "--time-budget-hours", type=float, default=10.5,
        help="Maximum runtime for this script; the Kaggle hard limit is about 12h.",
    )
    parser.add_argument("--imgsz", type=int, default=640)
    parser.add_argument("--batch", type=int, default=16,
                        help="Total batch size; lower to 8 if a GPU runs out of memory.")
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--device", default="auto", help="auto, cpu, 0, or 0,1")
    parser.add_argument("--patience", type=int, default=20)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--resume", action="store_true")
    args, _ = parser.parse_known_args(argv)
    started = time.monotonic()

    if (args.epochs < 1 or args.imgsz < 32 or args.workers < 0
            or args.batch < 1 or args.time_budget_hours <= 0):
        raise ValueError(
            "--epochs, --imgsz, --batch, and --time-budget-hours must be positive; "
            "--workers cannot be negative."
        )
    if not args.model.lower().endswith("-seg.pt"):
        raise ValueError(
            f"Expected an instance-segmentation checkpoint such as {MODEL_NAME}; got {args.model!r}."
        )

    _ensure_dependencies()
    from ultralytics import YOLO

    root = _resolve_dataset_root(args.data_root)
    names, split_dirs = _load_dataset_yaml(root)
    image_counts = _validate_labels(root, names, split_dirs)
    image_annotation_cv = names == IMAGE_ANNOTATION_CLASSES
    cv_items = _collect_cv_items(
        root, split_dirs, group_filename_augmentations=image_annotation_cv
    )
    fold_indices, split_report = _assign_folds(cv_items, len(names), args.seed)
    device = _select_device(args.device)
    work_dir = (
        Path(args.work_dir).expanduser().resolve()
        if args.work_dir
        else Path("/kaggle/working/session_34_yolo26n_seg_image_annotation")
        if Path("/kaggle/working").is_dir()
        else Path.cwd() / "session_34_yolo26n_seg_image_annotation"
    )
    work_dir.mkdir(parents=True, exist_ok=True)
    yaml_paths = _write_cv_fold_yamls(
        work_dir, cv_items, fold_indices, names, args.seed
    )
    plan_signature = hashlib.sha256(
        json.dumps(
            {
                "items": cv_items,
                "fold_indices": fold_indices,
                "seed": args.seed,
                "names": names,
            },
            sort_keys=True,
        ).encode("utf-8")
    ).hexdigest()
    plan_path = work_dir / "cv_split_report.json"
    if args.resume and plan_path.is_file():
        saved_plan = json.loads(plan_path.read_text(encoding="utf-8"))
        if saved_plan.get("split_signature") != plan_signature:
            raise RuntimeError(
                "The saved CV split differs from the current dataset or seed. "
                "Use the original dataset and --seed, or choose a new --work-dir."
            )
    split_report["split_signature"] = plan_signature
    split_report["folds"] = [
        {
            "fold": fold_id,
            "validation_images": split_report["fold_validation_image_counts"][fold_id],
            "training_images": len(cv_items) - split_report["fold_image_counts"][fold_id],
            "validation_source_groups": split_report["fold_group_counts"][fold_id],
            "all_held_out_group_variants": split_report["fold_image_counts"][fold_id],
            "yaml": str(yaml_path),
        }
        for fold_id, yaml_path in enumerate(yaml_paths)
    ]
    dataset_summary_path = root / "summary.json"
    dataset_evaluation_note = None
    if dataset_summary_path.is_file():
        dataset_summary = json.loads(
            dataset_summary_path.read_text(encoding="utf-8")
        )
        dataset_evaluation_note = dataset_summary.get("evaluation_note")
    if image_annotation_cv:
        overlap_count = cv_items[0]["source_split_overlap_groups"]
        conflict_count = cv_items[0]["excluded_source_class_conflict_groups"]
        dataset_evaluation_note = (
            "The original train/valid/test splits contain augmentation variants of the same "
            f"filename-derived source groups ({overlap_count} groups span multiple splits). "
            "All splits are therefore pooled for source-grouped five-fold CV; each remaining "
            "source group is held out as a unit, and validation uses one unaugmented-preferred "
            "image per group. The original test split is not an independent test set. "
            f"{conflict_count} source groups with class-set disagreement across variants were "
            "excluded. No patient IDs are available in this dataset."
        )
        split_report["original_split_source_overlap_groups"] = overlap_count
        split_report["excluded_source_class_conflict_groups"] = conflict_count
        split_report["validation_image_policy"] = (
            "one non-augmentation-prefixed image per source group; if unavailable, "
            "one deterministic augmentation variant"
        )
    plan_path.write_text(json.dumps(split_report, indent=2), encoding="utf-8")

    effective_batch = args.batch
    pending = []
    fold_results: list[dict[str, Any]] = []
    for fold_id, fold_yaml in enumerate(yaml_paths):
        run_name = f"seed_{args.seed}_fold_{fold_id}"
        run_dir = work_dir / "runs" / run_name
        best_checkpoint = run_dir / "weights" / "best.pt"
        last_checkpoint = run_dir / "weights" / "last.pt"
        results_csv = run_dir / "results.csv"
        fold_metrics_path = work_dir / "metrics" / f"{run_name}.json"
        fold_signature = hashlib.sha256(
            json.dumps(
                {
                    "split_signature": plan_signature,
                    "model": args.model,
                    "epochs": args.epochs,
                    "imgsz": args.imgsz,
                    "batch": args.batch,
                    "patience": args.patience,
                    "seed": args.seed,
                    "fold": fold_id,
                },
                sort_keys=True,
            ).encode("utf-8")
        ).hexdigest()
        if fold_metrics_path.is_file():
            saved = json.loads(fold_metrics_path.read_text(encoding="utf-8"))
            if saved.get("fold_signature") != fold_signature:
                raise RuntimeError(
                    f"Fold {fold_id} artifacts use different training settings. "
                    "Choose a new --work-dir to avoid mixing results."
                )
            if not best_checkpoint.is_file():
                raise FileNotFoundError(
                    f"Fold {fold_id} metrics exist but checkpoint is missing: "
                    f"{best_checkpoint}"
                )
            fold_results.append(saved)
            print(f"[fold {fold_id}] reusing completed fold.", flush=True)
        else:
            pending.append((fold_id, fold_yaml))

    print(f"[data] root={root}", flush=True)
    if dataset_evaluation_note:
        print(f"[evaluation note] {dataset_evaluation_note}", flush=True)
    if image_annotation_cv:
        print(
            f"[data] all splits pooled={len(cv_items)} images; "
            f"source groups={split_report['source_group_count']}; "
            f"validation representatives={sum(split_report['fold_validation_image_counts'])}; "
            f"classes={len(names)}",
            flush=True,
        )
    else:
        print(
            f"[data] source train+valid={len(cv_items)} images; external test="
            f"{image_counts['test']} images (reserved, not used in CV); "
            f"classes={len(names)}",
            flush=True,
        )
    print(
        f"[cv] folds={FOLD_COUNT}; train runs={FOLD_COUNT}; "
        f"epochs argument/run={args.epochs} (wall-time limit takes precedence); "
        f"fold held-out variant images={split_report['fold_group_variant_image_counts']}; "
        f"validation images={split_report['fold_validation_image_counts']}",
        flush=True,
    )
    print(
        f"[cv] image-passes at {args.epochs} epochs/run="
        f"{sum(len(cv_items) - count for count in split_report['fold_image_counts']) * args.epochs:,}; "
        f"script time budget={args.time_budget_hours:.2f}h; "
        "45 minutes are reserved for final fold evaluations and reporting.",
        flush=True,
    )
    print(f"[train] model={args.model}; device={device}; batch={effective_batch}", flush=True)

    for pending_position, (fold_id, fold_yaml) in enumerate(pending):
        remaining_fold_count = len(pending) - pending_position
        elapsed_hours = (time.monotonic() - started) / 3600.0
        remaining_hours = args.time_budget_hours - elapsed_hours
        reserved_hours = 0.75
        train_hours = (remaining_hours - reserved_hours) / remaining_fold_count
        if train_hours <= 0.05:
            raise TimeoutError(
                f"Not enough of the {args.time_budget_hours:.2f}h budget remains "
                f"to train and evaluate all {remaining_fold_count} unfinished folds. "
                "Use a smaller --epochs value or a fresh Kaggle session."
            )

        run_name = f"seed_{args.seed}_fold_{fold_id}"
        run_dir = work_dir / "runs" / run_name
        best_checkpoint = run_dir / "weights" / "best.pt"
        last_checkpoint = run_dir / "weights" / "last.pt"
        fold_metrics_path = work_dir / "metrics" / f"{run_name}.json"
        if last_checkpoint.is_file() and not args.resume:
            raise RuntimeError(
                f"Incomplete checkpoint exists for fold {fold_id}: {last_checkpoint}. "
                "Rerun with --resume or use a new --work-dir."
            )
        resume_fold = args.resume and last_checkpoint.is_file()
        model = YOLO(str(last_checkpoint)) if resume_fold else YOLO(args.model)
        fold_started = time.monotonic()
        print(
            f"[fold {fold_id}] train="
            f"{len(cv_items) - split_report['fold_image_counts'][fold_id]}, "
            f"valid={split_report['fold_validation_image_counts'][fold_id]}, "
            f"held_out_variants={split_report['fold_group_variant_image_counts'][fold_id]}, "
            f"epochs_arg={args.epochs}, time_limit={train_hours:.2f}h, "
            f"resume={resume_fold}",
            flush=True,
        )
        model.train(
            data=str(fold_yaml),
            task="segment",
            epochs=args.epochs,
            time=train_hours,
            imgsz=args.imgsz,
            batch=effective_batch,
            workers=args.workers,
            device=device,
            patience=args.patience,
            seed=args.seed + fold_id,
            project=str(work_dir / "runs"),
            name=run_name,
            exist_ok=True,
            resume=resume_fold,
            amp=True,
            plots=True,
            save=True,
            verbose=True,
        )
        del model
        if not best_checkpoint.is_file():
            raise FileNotFoundError(
                f"Fold {fold_id} training ended without best.pt: {best_checkpoint}"
            )

        evaluation_model = YOLO(str(best_checkpoint))
        evaluation_device = device.split(",")[0].strip() if device not in ("cpu", "") else "cpu"
        metrics = evaluation_model.val(
            data=str(fold_yaml),
            split="val",
            imgsz=args.imgsz,
            batch=min(effective_batch, 8),
            workers=args.workers,
            device=evaluation_device,
            plots=True,
            project=str(work_dir / "runs"),
            name=f"{run_name}_evaluation",
            exist_ok=True,
            verbose=True,
        )
        fold_result = {
            "fold": fold_id,
            "fold_signature": hashlib.sha256(
                json.dumps(
                    {
                        "split_signature": plan_signature,
                        "model": args.model,
                        "epochs": args.epochs,
                        "imgsz": args.imgsz,
                        "batch": args.batch,
                        "patience": args.patience,
                        "seed": args.seed,
                        "fold": fold_id,
                    },
                    sort_keys=True,
                ).encode("utf-8")
            ).hexdigest(),
            "metrics": _segmentation_results(metrics, names),
            "best_checkpoint": str(best_checkpoint),
            "epochs_completed": _completed_epoch_count(results_csv),
            "elapsed_seconds": time.monotonic() - fold_started,
        }
        fold_metrics_path.parent.mkdir(parents=True, exist_ok=True)
        fold_metrics_path.write_text(
            json.dumps(fold_result, indent=2), encoding="utf-8"
        )
        _write_per_class_csv(
            work_dir / "metrics" / f"{run_name}_per_class_segm_map50.csv",
            fold_result["metrics"]["per_class_segm_mAP50"],
        )
        fold_results.append(fold_result)
        print(
            f"[fold {fold_id}] segm mAP50="
            f"{fold_result['metrics']['segm_mAP50']}; "
            f"epochs={fold_result['epochs_completed']}; "
            f"elapsed={(time.monotonic() - fold_started) / 60.0:.1f}min",
            flush=True,
        )

    fold_results.sort(key=lambda result: result["fold"])
    cv_metrics = _aggregate_fold_metrics(fold_results, names)
    _write_per_class_csv(
        work_dir / "cv_per_class_segm_map50.csv",
        {
            name: values["mean"]
            for name, values in cv_metrics["per_class_segm_mAP50"].items()
        },
    )
    summary = {
        "session": SESSION_NAME,
        "model": args.model,
        "task": "instance segmentation",
        "dataset_root": str(root),
        "dataset_evaluation_note": dataset_evaluation_note,
        "class_names": names,
        "image_counts": image_counts,
        "cross_validation": {
            "folds": FOLD_COUNT,
            "fold_seed": args.seed,
            "training_splits": (
                "all original train/valid/test pooled for filename-source-grouped CV; "
                "the original test split is not independent"
                if image_annotation_cv
                else "original train + valid pooled; external test excluded"
            ),
            "split_report": split_report,
            "fold_results": fold_results,
            "aggregate_metrics": cv_metrics,
        },
        "training_plan": {
            "epochs_argument_per_fold": args.epochs,
            "epoch_argument_note": "Ultralytics time limit takes precedence when set",
            "image_size": args.imgsz,
            "batch": effective_batch,
            "device": device,
            "seed": args.seed,
            "maximum_training_runs": FOLD_COUNT,
            "nominal_image_passes_at_epoch_argument": (
                sum(
                    len(cv_items) - count
                    for count in split_report["fold_image_counts"]
                ) * args.epochs
            ),
            "completed_epochs_by_fold": {
                str(result["fold"]): result.get("epochs_completed", 0)
                for result in fold_results
            },
            "actual_training_image_passes": sum(
                (len(cv_items) - split_report["fold_image_counts"][result["fold"]])
                * result.get("epochs_completed", 0)
                for result in fold_results
            ),
            "time_budget_hours": args.time_budget_hours,
            "elapsed_seconds": time.monotonic() - started,
            "fold_time_limit_hours": "dynamically allocated within the total time budget",
        },
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    summary_path = work_dir / "SESSION_34_results.json"
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(
        f"[cv] mean segmentation mAP50="
        f"{cv_metrics['segm_mAP50']['mean']} +/- "
        f"{cv_metrics['segm_mAP50']['std']}",
        flush=True,
    )
    print(
        "[cv] completed epochs="
        f"{sum(result.get('epochs_completed', 0) for result in fold_results)}"
        f"/{FOLD_COUNT * args.epochs} maximum",
        flush=True,
    )
    print("[cv] per-class mean segmentation mAP50:", flush=True)
    for name, values in cv_metrics["per_class_segm_mAP50"].items():
        print(f"  {name}: {values['mean']} (std={values['std']})", flush=True)
    print(f"[complete] results={summary_path}", flush=True)
    print(f"[complete] fold checkpoints={work_dir / 'runs'}", flush=True)
    return 0


if __name__ == "__main__":
    multiprocessing.freeze_support()
    raise SystemExit(main())


[install] Ensuring the latest Ultralytics package for YOLO26 support.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 5.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[environment] ultralytics=8.4.175
[device] using GPU(s) 0,1 of 2 available.
[data] root=/kaggle/input/datasets/muntasirulmunna/image-annotation-rebuilt
[evaluation note] The original train/valid/test splits contain augmentation variants of the same filename-derived source groups (316 groups span multiple splits). All splits are therefore pooled for source-grouped five-fold CV; each remaining source 

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       2/48      1.89G      1.334      2.409       1.85    0.01981      2.855          9        640: 100% ━━━━━━━━━━━━ 401/401 3.6it/s 1:51
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.7it/s 0.4s
                   all         62        122      0.609      0.321       0.37      0.186      0.542      0.292      0.324      0.139

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       3/54      1.89G      1.386      2.427      1.828    0.02117      2.776          9        640: 100% ━━━━━━━━━━━━ 401/401 3.8it/s 1:46
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 5.0it/s 0.4s
                   all         62        122   

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       2/52      1.99G      1.317       2.48      1.868    0.02013      2.839         31        640: 100% ━━━━━━━━━━━━ 402/402 3.7it/s 1:49
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.7it/s 0.4s
                   all         62        128      0.634      0.475      0.502       0.27      0.593      0.435      0.432      0.232

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       3/59      1.99G      1.394      2.498      1.836    0.02174      2.726         26        640: 100% ━━━━━━━━━━━━ 402/402 3.7it/s 1:48
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.9it/s 0.4s
                   all         62        128   

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


... (3 outputs left out)

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       2/80      1.88G      1.315      2.402      1.883    0.02024      2.844         30        640: 100% ━━━━━━━━━━━━ 401/401 3.7it/s 1:49
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.8it/s 0.4s
                   all         62        135      0.589      0.447      0.423      0.237      0.521      0.438      0.395      0.209

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss    l1_loss   sem_loss  Instances       Size
       3/90      1.88G      1.375      2.424      1.905    0.02129      2.872         34        640: 100% ━━━━━━━━━━━━ 401/401 3.7it/s 1:47
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.7it/s 0.4s
                   all         62        135   

SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
